# 04 · Encapsulation and abstraction

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 1 | about 60 minutes live | [03 · Special methods](03_special_methods.ipynb) | [05 · Polymorphism, duck typing and protocols](05_polymorphism_duck_typing_protocols.ipynb) |


Many objects in data science **learn** something from data and then **reuse** it: a scaler learns means, an encoder learns categories, a model learns weights. What an object has learned is its **state**, and two ideas keep that state trustworthy:

- **Encapsulation**: an object controls its own state. Callers use a small public interface and do not reach into the internals.
- **Abstraction**: a base class states *what* every member of a family must be able to do (for preprocessing steps: `fit` and `transform`) without saying *how*.

In the hands-on parts you use both ideas to solve the mystery of the warm-up notebook (00).

> **Learning goals.** By the end you can:
>
> - explain why a preprocessing step must learn its parameters in `fit` and reuse them in `transform`
> - separate public interface from internal state with naming conventions, read-only properties and validating setters
> - define an abstract base class and implement a subclass that fulfils its contract
> - recognise data that leaks out of an object through shared references, and prevent it

**Before you start:** notebooks 01 to 03.

**In this notebook**

- Theory 1 · Learned state: `fit` and `transform`
- Theory 2 · Encapsulation: public interface, internal state
- Theory 3 · Properties with validation
- Hands-on A · The warm-up mystery, solved
- Theory 4 · Abstraction: abstract base classes
- Hands-on B · Preprocessing steps with a shared contract

---

## Setup

Run this cell first. It loads the churn data that the hands-on parts use; the theory parts use small hand-made examples.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/telco_churn.csv")

customers = pd.read_csv(DATA_PATH).set_index("customerID")  # the customer ID becomes the row label
# .pop removes the column and returns it; == "Yes" gives True/False; .astype(int) turns that into 1/0
churned = (customers.pop("Churn") == "Yes").astype(int)

CATEGORICAL = ["Contract", "InternetService", "PaymentMethod"]
NUMERIC = ["tenure", "MonthlyCharges"]

train_customers, new_customers, y_train, y_new = train_test_split(
    customers, churned, test_size=0.2, random_state=42, stratify=churned
)
print(
    f"{len(train_customers)} customers to learn from, {len(new_customers)} 'new' customers to score"
)

---

## Theory 1 · Learned state: `fit` and `transform`

Some preprocessing steps only need the value in front of them: taking the logarithm of a price works the same on any row. Others must first **learn** something from data. Standardising a column (subtract the mean, divide by the standard deviation) needs the mean and the standard deviation. Turning a text column into numbers needs the list of categories.

Such steps are written as classes with two methods, the pattern scikit-learn uses everywhere:

- `fit(train)` computes the parameters from the training data and **stores them in the object** as attributes, then returns the object;
- `transform(batch)` applies the stored parameters to any batch and learns nothing.

What the object stored is its **state**. Here is a small standardiser.

In [ ]:
class Standardizer:
    """Scale columns to mean 0 and standard deviation 1, using training statistics."""

    def __init__(self, columns: list[str]):
        self.columns = list(columns)
        self.means = {}  # learned in fit: column -> mean
        self.stds = {}  # learned in fit: column -> standard deviation
        self.fitted = False

    def fit(self, X: pd.DataFrame) -> "Standardizer":
        for col in self.columns:
            self.means[col] = float(X[col].mean())  # float(...) gives a plain Python number
            self.stds[col] = float(X[col].std())
        self.fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self.fitted:
            raise RuntimeError("Standardizer must be fitted before transform()")
        out = X.copy()  # a new DataFrame, so the caller's data stays unchanged
        for col in self.columns:
            out[col] = (X[col] - self.means[col]) / self.stds[col]
        return out


train_charges = pd.DataFrame({"MonthlyCharges": [20.0, 50.0, 80.0, 110.0]})
new_charges = pd.DataFrame({"MonthlyCharges": [100.0, 110.0]})

scaler = Standardizer(["MonthlyCharges"]).fit(train_charges)
print("learned means:", scaler.means)
print("learned standard deviations:", scaler.stds)
print(scaler.transform(new_charges).round(2))

Why must `transform` never learn? Fit a second standardiser on the new batch and compare:

In [ ]:
refitted = Standardizer(["MonthlyCharges"]).fit(new_charges)
print(refitted.transform(new_charges).round(2))

A monthly charge of 110 becomes 1.16 with the training statistics and 0.71 with the statistics of the new batch. A model trained on the first scale would read the second number as a much cheaper customer. The rule: **fit once, on training data; afterwards only transform.**

That is why `fit` and `transform` are separate methods, and why the learned values must be kept safe. Keeping them safe is what encapsulation is about.

---

## Theory 2 · Encapsulation: public interface, internal state

Every attribute of `Standardizer` is public: anyone who holds the object can read and change `scaler.means`. Here is how that goes wrong, without anybody doing something obviously silly. A colleague needs the average monthly charge for a slide and rounds it:

In [ ]:
report = scaler.means  # "I only read the averages for a slide"
report["MonthlyCharges"] = round(report["MonthlyCharges"], -1)  # 65.0 rounded to tens

print("scaler.means is now:", scaler.means)
print(scaler.transform(new_charges).round(2))

The scaler's output changed. `report` was not a copy: it is the **same dict object** the scaler uses internally. Python passes references, so whoever holds a reference to an object's internals can change its state.

Encapsulation addresses this in two steps:

1. **Mark internals as internal.** A leading underscore (`_means`) is Python's convention for "implementation detail, do not use from outside". It is a convention, not a lock: Python still lets you access `scaler._means`. But it tells every reader and every editor that this attribute is not part of the interface and may change without notice.
2. **Expose what callers legitimately need through a controlled interface.** A **property** (notebook 01) looks like an attribute to the caller but runs a method. Here it returns a *copy*, so nobody can corrupt the original by accident.

In [ ]:
class Standardizer:
    """Scale columns to mean 0 and standard deviation 1, using training statistics."""

    def __init__(self, columns: list[str]):
        self.columns = list(columns)
        self._means: dict[str, float] = {}
        self._stds: dict[str, float] = {}
        self._fitted = False

    @property
    def means(self) -> dict[str, float]:
        """Learned means per column (a copy: changing it cannot affect the scaler)."""
        return dict(self._means)  # dict(...) builds a new dict with the same keys and values

    @property
    def is_fitted(self) -> bool:
        return self._fitted

    def fit(self, X: pd.DataFrame) -> "Standardizer":
        for col in self.columns:
            self._means[col] = float(X[col].mean())
            self._stds[col] = float(X[col].std())
        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("Standardizer must be fitted before transform()")
        out = X.copy()
        for col in self.columns:
            out[col] = (X[col] - self._means[col]) / self._stds[col]
        return out


scaler = Standardizer(["MonthlyCharges"]).fit(train_charges)
report = scaler.means
report["MonthlyCharges"] = round(report["MonthlyCharges"], -1)  # the same slide code as before

print("slide shows:      ", report)
print("scaler still uses:", scaler.means)
print(scaler.transform(new_charges).round(2))

Read-only properties also refuse assignment, which catches the "I'll just set it quickly" mistake:

In [ ]:
try:
    scaler.is_fitted = False
except AttributeError as exc:
    print("AttributeError:", exc)

### What about double underscores?

`self.__password` (two leading underscores) triggers **name mangling**: Python renames the attribute to `_ClassName__password`. Its purpose is to avoid name clashes in subclasses, not to hide data. It is rarely needed in data science code; a single underscore is the norm.

In [ ]:
class WarehouseConnection:
    def __init__(self, user: str, password: str):
        self.user = user
        self.__password = password


conn = WarehouseConnection("analyst", "s3cret")
print(vars(conn))  # vars(obj) shows the attributes stored on the object: the name was changed
print(conn._WarehouseConnection__password)  # and the value is still reachable

So double underscores do not keep a secret. Real secrets (passwords, API keys) belong in environment variables or a secrets manager, never in code.

> **Check your understanding**
>
> Which of these lines does Python prevent?
>
> ```python
> scaler._means = {}                      # (a)
> scaler.means = {}                       # (b)
> scaler.means["MonthlyCharges"] = 0      # (c)
> ```
>
> <details><summary><b>Show answer</b></summary>
>
> Only (b) raises: `means` is a property without a setter, so assignment fails with `AttributeError`. Line (a) works: the underscore is a convention, and the person who writes it has been warned. Line (c) runs without error, but it only changes the *copy* that the property returned, so the scaler is unaffected. Encapsulation in Python is about making the right thing easy and the wrong thing obviously wrong, not about locks.
> </details>

In [ ]:
# Predict the output first, then run this cell
demo = Standardizer(["MonthlyCharges"]).fit(train_charges)

demo._means = {}  # (a)
print("(a) ran without error")

try:
    demo.means = {}  # (b)
    print("(b) ran without error")
except AttributeError as exc:
    print("(b) AttributeError:", exc)

demo = Standardizer(["MonthlyCharges"]).fit(train_charges)  # a fresh scaler for (c)
demo.means["MonthlyCharges"] = 0  # (c)
print("(c) ran without error; the scaler still uses:", demo.means)

---

## Theory 3 · Properties with validation

A property can also have a **setter**: a method that runs when someone assigns to the attribute. That is the right place to protect a rule the object must always satisfy (an *invariant*).

Example: a classifier outputs probabilities, and a **decision threshold** turns them into yes/no decisions. Changing the threshold after training is a legitimate business decision, so it should be settable. But it must stay a probability: a threshold of `30` (someone typed a percentage) would silently flag nobody.

In [ ]:
class ThresholdFlagger:
    """Turn probabilities into 0/1 decisions with an adjustable threshold."""

    def __init__(self, threshold: float = 0.5):
        self.threshold = threshold  # goes through the setter below, so it is validated too

    @property
    def threshold(self) -> float:
        return self._threshold

    @threshold.setter
    def threshold(self, value: float) -> None:
        if not 0 < value < 1:
            raise ValueError(f"threshold must be a probability between 0 and 1, got {value}")
        self._threshold = value

    def flag(self, probabilities: np.ndarray) -> np.ndarray:
        # probabilities >= threshold gives an array of True/False; .astype(int) turns it into 1/0
        return (probabilities >= self._threshold).astype(int)


probabilities = np.array([0.12, 0.35, 0.58, 0.81])
flagger = ThresholdFlagger()
print("threshold 0.5:", flagger.flag(probabilities))

flagger.threshold = 0.3
print("threshold 0.3:", flagger.flag(probabilities))

try:
    flagger.threshold = 30  # meant 30 %
except ValueError as exc:
    print("ValueError:", exc)

Notice that `__init__` assigns `self.threshold`, not `self._threshold`. That way the constructor uses the same validation as every later assignment, and there is exactly one place where the rule lives.

---

## Hands-on A · The warm-up mystery, solved

Back to the churn project. In the warm-up notebook (00), scoring next month's customers gave different results depending on who else was in the file. With Theory 1 you can name the cause. Here is the bug in miniature: four training customers, and next month a batch of two customers who both happen to have a two-year contract.

In [ ]:
from sklearn.preprocessing import LabelEncoder

train = pd.DataFrame({"Contract": ["Month-to-month", "One year", "Two year", "Month-to-month"]})
next_month = pd.DataFrame({"Contract": ["Two year", "Two year"]})

encoder = LabelEncoder()
train_codes = encoder.fit_transform(train["Contract"])
print("training:  ", list(train["Contract"]), "->", train_codes.tolist())
next_codes = encoder.fit_transform(next_month["Contract"])  # fitted again, on the new batch
print("next month:", list(next_month["Contract"]), "->", next_codes.tolist())

print()
# pd.get_dummies creates one 0/1 column per category it finds in the data it is given
print("get_dummies at training time:", pd.get_dummies(train["Contract"]).columns.tolist())
print("get_dummies next month:      ", pd.get_dummies(next_month["Contract"]).columns.tolist())

"Two year" was code 2 when the model was trained and becomes code 0 next month, because the encoder was fitted again on the new batch and saw only one category. The model receives a number it interprets as "Month-to-month". No error, no warning, wrong predictions. `get_dummies` has the same root cause; there the set of columns changes, which is at least easier to notice.

The encoding is learned state, and it was learned from the wrong data. The fix follows Theory 1 and 2: an encoder that learns its categories in `fit` (on training data only), keeps them internal, and only applies them in `transform`.

**Worked example.** Read and run this encoder; the exercises below build on the same ideas.

In [ ]:
class OneHotEncoder:
    """One 0/1 column per category seen during fit. Same output columns for every batch."""

    def __init__(self, columns: list[str]):
        self.columns = list(columns)
        self._categories: dict[str, list[str]] = {}
        self._fitted = False

    @property
    def categories(self) -> dict[str, list[str]]:
        """Learned categories per column (a copy: changing it cannot affect the encoder)."""
        return {col: list(cats) for col, cats in self._categories.items()}

    @property
    def output_columns(self) -> list[str]:
        return [f"{col}={cat}" for col, cats in self._categories.items() for cat in cats]

    @property
    def is_fitted(self) -> bool:
        return self._fitted

    def fit(self, X: pd.DataFrame) -> "OneHotEncoder":
        # X[col].unique() lists the distinct values of a column
        self._categories = {col: sorted(X[col].unique()) for col in self.columns}
        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("OneHotEncoder must be fitted before transform()")
        out = X.drop(columns=self.columns)  # a new DataFrame without the text columns
        for col, cats in self._categories.items():
            for cat in cats:
                # (X[col] == cat) is True/False per row; .astype(int) makes it 1/0
                out[f"{col}={cat}"] = (X[col] == cat).astype(int)
        return out

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)


encoder = OneHotEncoder(["Contract"]).fit(train)
encoder.transform(next_month)

The two-year customers now get exactly the columns the model was trained with. Now on the real data: train a model with this encoder, then score the two-year customers **alone** (a batch where `Contract` has only one value) and compare with their scores **inside the full batch**.

In [ ]:
from sklearn.linear_model import LogisticRegression

churn_encoder = OneHotEncoder(CATEGORICAL).fit(train_customers)


def model_input(batch: pd.DataFrame) -> pd.DataFrame:
    """Columns the model sees: numeric features plus the learned one-hot columns."""
    return churn_encoder.transform(batch)[NUMERIC + churn_encoder.output_columns]


model = LogisticRegression(max_iter=1000).fit(model_input(train_customers), y_train)

two_year = new_customers[
    new_customers["Contract"] == "Two year"
]  # rows where the condition is True

# predict_proba returns one column per class; [:, 1] takes column 1, the probability of churn
alone = model.predict_proba(model_input(two_year))[:, 1]
in_full_batch = pd.Series(
    model.predict_proba(model_input(new_customers))[:, 1], index=new_customers.index
)[two_year.index]

print(f"{len(two_year)} two-year customers")
print("identical scores alone and in the full batch:", np.allclose(alone, in_full_batch))

A customer's score no longer depends on who else is in the file. In notebook 10 this comparison becomes an automated regression test.

The `ThresholdFlagger` from Theory 3 turns these scores into a call list for the retention team:

In [ ]:
flagger = ThresholdFlagger(threshold=0.3)
churn_probability = model.predict_proba(model_input(new_customers))[:, 1]
print(f"{flagger.flag(churn_probability).sum()} of {len(new_customers)} customers to call")

### Exercise A.1: a `MedianImputer`

Next month's batch has gaps: some customers have no `MonthlyCharges` because of a billing-system migration. You must fill them, and the fill value must come from the **training data**. If you used the median of the new batch, a batch of mostly expensive fibre customers would get a different fill value than a batch of mostly cheap DSL customers: the same bug as before, in a new costume.

Write a class `MedianImputer`, following the `Standardizer` and `OneHotEncoder` above.

**What your code should do.**

- `MedianImputer(["MonthlyCharges"])` creates an unfitted imputer for the listed columns.
- `fit(X)` takes the training DataFrame, stores one median per listed column internally, and returns the imputer itself, so `MedianImputer([...]).fit(train_customers)` works.
- `transform(X)` returns a **new** DataFrame with the same rows and columns as `X`, where every missing value (`NaN`) in the listed columns is replaced by the training median. Other columns, and `X` itself, stay unchanged. Called before `fit`, it raises `RuntimeError`.
- `imputer.medians` is a read-only property returning a dict such as `{"MonthlyCharges": 70.5}`; changing that dict must not change the imputer.

**pandas you need.**

- `X["MonthlyCharges"].median()`: the median of a column, one number (missing values are ignored). Wrap it in `float(...)` to store a plain Python number.
- `X.copy()`: a new DataFrame with the same content, so changes do not affect `X`.
- `out["MonthlyCharges"] = out["MonthlyCharges"].fillna(70.5)`: replace the missing values of one column by a number.

**Discuss.** Your imputer always fills with the *training* median. When could that value itself be wrong for next month's batch (think of a price increase)? Would you rather fill the gaps, flag those customers, or refuse the whole batch, and who should decide?

In [ ]:
# Exercise A.1: your code here


class MedianImputer:
    def __init__(self, columns: list[str]):
        raise NotImplementedError("Your code here")

    @property
    def medians(self) -> dict[str, float]:
        raise NotImplementedError("Your code here")

    def fit(self, X: pd.DataFrame) -> "MedianImputer":
        raise NotImplementedError("Your code here")

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        raise NotImplementedError("Your code here")


batch = new_customers.head(6).copy()
batch.loc[batch.index[[1, 4]], "MonthlyCharges"] = np.nan  # make two values missing

imputer = MedianImputer(["MonthlyCharges"]).fit(train_customers)
imputer.transform(batch)[["Contract", "MonthlyCharges"]]

In [ ]:
# Check for exercise A.1: run after solving it
train_median = train_customers["MonthlyCharges"].median()
filled = imputer.transform(batch)
assert filled["MonthlyCharges"].isna().sum() == 0
assert filled["MonthlyCharges"].iloc[1] == train_median  # the training median ...
assert filled["MonthlyCharges"].iloc[4] == train_median
assert batch["MonthlyCharges"].isna().sum() == 2  # ... and the input is untouched
assert imputer.medians == {"MonthlyCharges": train_median}
imputer.medians["MonthlyCharges"] = 0  # changing the returned dict ...
assert imputer.medians == {"MonthlyCharges": train_median}  # ... must not change the imputer
try:
    MedianImputer(["MonthlyCharges"]).transform(batch)
    raise AssertionError("transform before fit should raise RuntimeError")
except RuntimeError:
    pass
print("All checks passed. Fill value:", train_median)

<details><summary><b>Show hint</b></summary>

Copy the structure of the encapsulated `Standardizer`: `_medians` and `_fitted` set in `__init__`, a loop over `self.columns` in `fit` and in `transform`, and a `medians` property that returns `dict(self._medians)`.
</details>

<details><summary><b>Show solution</b></summary>

```python
class MedianImputer:
    def __init__(self, columns: list[str]):
        self.columns = list(columns)
        self._medians: dict[str, float] = {}
        self._fitted = False

    @property
    def medians(self) -> dict[str, float]:
        return dict(self._medians)

    def fit(self, X: pd.DataFrame) -> "MedianImputer":
        for col in self.columns:
            self._medians[col] = float(X[col].median())
        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("MedianImputer must be fitted before transform()")
        out = X.copy()
        for col in self.columns:
            out[col] = out[col].fillna(self._medians[col])
        return out
```
</details>

### Exercise A.2: what about categories nobody has seen?

> If you finish early, try this one.

The company launches a "Three year" contract. `OneHotEncoder` silently encodes such customers as all zeros in every `Contract=...` column. Sometimes that is acceptable (the model treats them like "none of the known contracts"), sometimes you would rather stop. scikit-learn's `OneHotEncoder` has a `handle_unknown` parameter for this choice. Write `SafeOneHotEncoder(OneHotEncoder)` and reuse the parent's logic with `super()`; do not copy the encoding loop.

**What your code should do.**

- `SafeOneHotEncoder(["Contract"])` behaves like `OneHotEncoder`, with one extra parameter `handle_unknown`: `"ignore"` (the default) or `"error"`. Any other value raises `ValueError` as soon as the object is created (fail early).
- `transform(X)` first looks for categories in `X` that were not seen during `fit`:
  - none: return exactly what `OneHotEncoder.transform(X)` returns;
  - some, with `"error"`: raise `ValueError` with a message naming the column and the unseen values, e.g. `Contract: categories not seen during fit: ['Three year']`;
  - some, with `"ignore"`: call `warnings.warn(...)` with the same message, then return the normal output (those rows get 0 in every `Contract=...` column).

`warnings.warn(message)` (from the standard-library module `warnings`) prints a warning and lets the code continue, unlike `raise`, which stops it. Notebook 09 compares warnings, logging and exceptions.

**pandas you need.**

- `X["Contract"].isin(["One year", "Two year"])`: a True/False value per row, True where the value is in the list.
- `~mask`: flips True and False (here: "not seen during fit").
- `X.loc[~mask, "Contract"].unique()`: the distinct values of the column in the rows where `~mask` is True; wrap it in `sorted(...)` to get a sorted list.

**Discuss.** For the churn model running every month, would you choose `"ignore"` or `"error"` as the default, and who needs to hear about a new contract type? Why might a library like scikit-learn choose `"error"` as its default instead?

In [ ]:
# Exercise A.2: your code here

import warnings


class SafeOneHotEncoder(OneHotEncoder):
    def __init__(self, columns: list[str], handle_unknown: str = "ignore"):
        raise NotImplementedError("Your code here")

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        raise NotImplementedError("Your code here")


launch = pd.DataFrame({"Contract": ["Three year", "One year"]})
safe = SafeOneHotEncoder(["Contract"]).fit(train)
# catch_warnings(record=True) collects the warnings in the list `caught` instead of only printing them,
# so the check cell below can read them. simplefilter("always") shows repeated warnings too.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    encoded = safe.transform(launch)
print([str(w.message) for w in caught])
encoded

In [ ]:
# Check for exercise A.2: run after solving it
assert len(caught) == 1 and "Three year" in str(caught[0].message)
assert encoded.loc[0].sum() == 0 and encoded.loc[1, "Contract=One year"] == 1
strict = SafeOneHotEncoder(["Contract"], handle_unknown="error").fit(train)
try:
    strict.transform(launch)
    raise AssertionError("expected ValueError for an unseen category")
except ValueError as exc:
    assert "Three year" in str(exc)
try:
    SafeOneHotEncoder(["Contract"], handle_unknown="skip")
    raise AssertionError("expected ValueError for an invalid handle_unknown")
except ValueError:
    pass
print("All checks passed")

<details><summary><b>Show hint</b></summary>

In `__init__`, validate first, then call `super().__init__(columns)` and store the option. In `transform`, `self._categories` tells you the known categories of each column. After the checks, `return super().transform(X)`.
</details>

<details><summary><b>Show solution</b></summary>

```python
class SafeOneHotEncoder(OneHotEncoder):
    def __init__(self, columns: list[str], handle_unknown: str = "ignore"):
        if handle_unknown not in ("ignore", "error"):
            raise ValueError(f"handle_unknown must be 'ignore' or 'error', got '{handle_unknown}'")
        super().__init__(columns)
        self.handle_unknown = handle_unknown

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.is_fitted:
            for col, known in self._categories.items():
                seen = X[col].isin(known)
                unseen = sorted(X.loc[~seen, col].unique())
                if not unseen:
                    continue
                message = f"{col}: categories not seen during fit: {unseen}"
                if self.handle_unknown == "error":
                    raise ValueError(message)
                warnings.warn(message + " (encoded as all zeros)", stacklevel=2)
        return super().transform(X)
```

`stacklevel=2` makes the warning point at the line that called `transform`, which is more useful to the reader than a line inside the encoder.

The `if self.is_fitted` guard lets the parent raise its usual "must be fitted" error. A subclass may read `self._categories`: the underscore means "internal to this class family", and subclasses are part of the family. `warnings.warn` (rather than `print`) lets the user of your class decide whether to show, log, silence, or turn the warning into an error.
</details>

---

## Theory 4 · Abstraction: abstract base classes

A real pipeline has several preprocessing steps: encode categories, fill gaps, scale numbers, cap outliers. They should all offer the same interface (`fit`, `transform`, `fit_transform`), so that the code using them does not care which step it has (that is polymorphism, notebook 05).

An **abstract base class** (ABC) writes this contract down:

- `@abstractmethod` marks methods every subclass *must* implement.
- Ordinary methods in the base class are shared by all subclasses. `fit_transform` is the same for every step, so it is written once.
- Python refuses to create an object of a class that still has abstract methods. A forgotten method fails at creation time, not halfway through a nightly job.

In [ ]:
from abc import ABC, abstractmethod


class Transformer(ABC):
    """Contract for a preprocessing step: learn in fit, replay in transform."""

    @abstractmethod
    def fit(self, X: pd.DataFrame) -> "Transformer":
        """Learn parameters from training data and return self."""

    @abstractmethod
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Apply the learned parameters. Must not learn anything, must not modify X."""

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)


try:
    Transformer()
except TypeError as exc:
    print("TypeError:", exc)


class HalfDoneScaler(Transformer):
    def fit(self, X):
        return self

    # transform() forgotten


try:
    HalfDoneScaler()
except TypeError as exc:
    print("TypeError:", exc)

A complete subclass implements both abstract methods and inherits `fit_transform`. This one caps extreme values at bounds learned from the training data, a common guard against data-entry errors:

In [ ]:
class ClipOutliers(Transformer):
    """Cap each column at its 1st and 99th percentile in the training data."""

    def __init__(self, columns: list[str]):
        self.columns = list(columns)
        self._low: dict[str, float] = {}
        self._high: dict[str, float] = {}

    def fit(self, X: pd.DataFrame) -> "ClipOutliers":
        for col in self.columns:
            self._low[col] = X[col].quantile(0.01)  # 1 % of the values lie below this one
            self._high[col] = X[col].quantile(0.99)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        out = X.copy()
        for col in self.columns:
            # .clip(low, high): values below low become low, values above high become high
            out[col] = X[col].clip(self._low[col], self._high[col])
        return out


train_totals = pd.DataFrame({"TotalCharges": [20.0, 150.0, 900.0, 2500.0, 8600.0]})
typos = pd.DataFrame({"TotalCharges": [-50.0, 3000.0, 250000.0]})  # a sign error and a typo

clipper = ClipOutliers(["TotalCharges"])
print(clipper.fit_transform(train_totals))  # fit_transform is inherited from Transformer
print(clipper.transform(typos))
print("is a Transformer:", isinstance(clipper, Transformer))

> **Check your understanding**
>
> ```python
> class LogTransform(Transformer):
>     def transform(self, X):
>         return np.log1p(X)
>
> step = LogTransform()
> ```
>
> What happens, and how would you fix it if the step really has nothing to learn?
>
> <details><summary><b>Show answer</b></summary>
>
> `TypeError` at `LogTransform()`: `fit` is still abstract. A step without parameters still has to honour the contract, so it implements a `fit` that learns nothing and returns `self`. That is exactly what scikit-learn's `FunctionTransformer` does. The contract makes every step interchangeable, even the trivial ones.
> </details>

In [ ]:
# Predict the output first, then run this cell
class LogTransform(Transformer):
    def transform(self, X):
        return np.log1p(X)  # log(1 + x), element by element


try:
    step = LogTransform()
    print("created:", step)
except TypeError as exc:
    print("TypeError:", exc)

---

## Hands-on B · Preprocessing steps with a shared contract

With the `Transformer` contract, every preprocessing step of the churn project can be used the same way. The encoder from Hands-on A joins the family: its first line changes, and `fit_transform` is now inherited instead of written again.

In [ ]:
class OneHotEncoder(Transformer):
    """One 0/1 column per category seen during fit. Same output columns for every batch."""

    def __init__(self, columns: list[str]):
        self.columns = list(columns)
        self._categories: dict[str, list[str]] = {}
        self._fitted = False

    @property
    def categories(self) -> dict[str, list[str]]:
        return {col: list(cats) for col, cats in self._categories.items()}

    @property
    def output_columns(self) -> list[str]:
        return [f"{col}={cat}" for col, cats in self._categories.items() for cat in cats]

    @property
    def is_fitted(self) -> bool:
        return self._fitted

    def fit(self, X: pd.DataFrame) -> "OneHotEncoder":
        self._categories = {col: sorted(X[col].unique()) for col in self.columns}
        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("OneHotEncoder must be fitted before transform()")
        out = X.drop(columns=self.columns)
        for col, cats in self._categories.items():
            for cat in cats:
                out[f"{col}={cat}"] = (X[col] == cat).astype(int)
        return out


churn_encoder = OneHotEncoder(CATEGORICAL).fit(train_customers)
print("is a Transformer:", isinstance(churn_encoder, Transformer))

### Exercise B.1: a step that removes columns

Some columns must never reach the churn model. `gender` and `SeniorCitizen` (age) describe who a customer is, not how they use the service. Using them to decide who gets a retention offer could be discriminatory, and in many countries it is not allowed. Write `DropColumns(Transformer)`.

**What your code should do.**

- `DropColumns(["gender", "SeniorCitizen"])` stores the columns to remove.
- `fit(X)` learns nothing and returns the object itself: the contract requires `fit` even when there is nothing to learn.
- `transform(X)` returns a new DataFrame without those columns; `X` itself stays unchanged.
- You do not write `fit_transform`: it comes from `Transformer`.

**pandas you need.**

- `X.drop(columns=["gender", "SeniorCitizen"])`: a new DataFrame without those columns (it raises `KeyError` if one of them does not exist).

**Discuss.** Dropping `gender` does not make a model fair by itself: other columns can carry similar information (so-called proxies). Which columns of this dataset could act as proxies for age or household situation? Where in the project should a rule like "never use these columns" live, and who should own it?

In [ ]:
# Exercise B.1: your code here


class DropColumns(Transformer):
    def __init__(self, columns: list[str]):
        raise NotImplementedError("Your code here")

    def fit(self, X: pd.DataFrame) -> "DropColumns":
        raise NotImplementedError("Your code here")

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise B.1: run after solving it
step = DropColumns(["gender", "SeniorCitizen"])
out = step.fit_transform(train_customers)
assert "gender" not in out.columns and "SeniorCitizen" not in out.columns
assert len(out.columns) == len(train_customers.columns) - 2
assert "gender" in train_customers.columns, "the input must stay unchanged"
assert isinstance(step, Transformer)
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
class DropColumns(Transformer):
    def __init__(self, columns: list[str]):
        self.columns = list(columns)

    def fit(self, X: pd.DataFrame) -> "DropColumns":
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return X.drop(columns=self.columns)
```

`drop` returns a new DataFrame, so the input stays unchanged without an explicit copy.
</details>

### Exercise B.2: a chain of steps that is itself a step

> **Challenge**

Write `TransformerChain(Transformer)`, constructed with a list of steps. This is the **Composite** pattern: a group of objects that can be used exactly like a single object.

**What your code should do.**

- `TransformerChain([step1, step2, ...])` stores the steps in order.
- `fit(X)` fits the first step on `X`, the second step on the output of the first, and so on (use each step's `fit_transform`), then returns the chain.
- `transform(X)` sends `X` through the already fitted steps in the same order and returns the final DataFrame. Called before `fit`, it raises `RuntimeError`.
- A chain is a `Transformer`, so a chain can be a step of another chain.

The cell below gives you a small `KeepColumns` step to build chains with.

**Discuss.** How does your class compare with scikit-learn's `Pipeline`? Name at least two things `Pipeline` does that yours does not. When would writing your own chain still be a reasonable choice?

In [ ]:
# Given: a step that keeps only some columns (nothing to learn)
class KeepColumns(Transformer):
    def __init__(self, columns: list[str]):
        self.columns = list(columns)

    def fit(self, X: pd.DataFrame) -> "KeepColumns":
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return X[self.columns]  # a DataFrame with only these columns

In [ ]:
# Exercise B.2: your code here


class TransformerChain(Transformer):
    def __init__(self, steps: list[Transformer]):
        raise NotImplementedError("Your code here")

    def fit(self, X: pd.DataFrame) -> "TransformerChain":
        raise NotImplementedError("Your code here")

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        raise NotImplementedError("Your code here")


chain = TransformerChain(
    [
        KeepColumns(["tenure", "Contract", "InternetService"]),
        OneHotEncoder(["Contract", "InternetService"]),
    ]
)
chain.fit(train_customers).transform(two_year).head(3)

In [ ]:
# Check for exercise B.2: run after solving it
kept = ["tenure", "Contract", "InternetService"]
expected = OneHotEncoder(["Contract", "InternetService"]).fit_transform(train_customers[kept])
pd.testing.assert_frame_equal(chain.fit_transform(train_customers), expected)
assert chain.transform(two_year).columns.equals(expected.columns)  # stable columns
assert isinstance(chain, Transformer)
nested = TransformerChain([chain, KeepColumns(["tenure", "Contract=Two year"])])
assert nested.fit_transform(train_customers).columns.tolist() == ["tenure", "Contract=Two year"]
try:
    TransformerChain([KeepColumns(["tenure"])]).transform(train_customers)
    raise AssertionError("transform before fit should raise RuntimeError")
except RuntimeError:
    pass
print("All checks passed")

<details><summary><b>Show hint</b></summary>

Keep `self._steps = list(steps)` and `self._fitted = False`. In `fit`, loop with a variable `current = X` and replace it by `step.fit_transform(current)` at each step. `transform` is the same loop with `step.transform`.
</details>

<details><summary><b>Show solution</b></summary>

```python
class TransformerChain(Transformer):
    def __init__(self, steps: list[Transformer]):
        self._steps = list(steps)
        self._fitted = False

    def fit(self, X: pd.DataFrame) -> "TransformerChain":
        current = X
        for step in self._steps:
            current = step.fit_transform(current)
        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("TransformerChain must be fitted before transform()")
        current = X
        for step in self._steps:
            current = step.transform(current)
        return current
```

Some things scikit-learn's `Pipeline` adds: the last step may be a model, so the whole pipeline has `predict` and `score`; steps have names and can be looked up (`pipe["scaler"]`, `pipe[0]`, `len(pipe)`, see notebook 06); `get_params`/`set_params` with `step__param` names, so `GridSearchCV` can tune parameters of inner steps; cloning for cross-validation; caching of fitted steps (`memory=`). Combined with `ColumnTransformer` it also applies different steps to different columns. In real projects, use `Pipeline`; writing your own once is how you understand it.
</details>

---

## Summary

- A preprocessing step **learns in `fit`, replays in `transform`**. Re-fitting on each new batch makes a customer's prediction depend on who else is in the file.
- Learned parameters are **state**. Keep it internal (`_name`), expose what callers need through **properties**, and hand out copies of mutable internals.
- A leading underscore is a convention, not a lock. It defines what is interface and what may change.
- **Validating setters** keep invariants (a threshold is a probability) in exactly one place; let `__init__` use the setter too.
- An **ABC** writes the contract down: forgetting a required method fails when the object is created, and shared behaviour (`fit_transform`) is written once.

**Common pitfalls**

- Returning a reference to an internal list or dict from a method or property.
- Computing statistics (medians, means, categories) on the batch being transformed instead of on the training data.
- Believing `_private` or `__private` makes data secret. It does not; it communicates intent.

## Going further

Not part of the live session.

- **Template method.** In our ABC every subclass repeats the "fitted?" bookkeeping. `src/churn/features.py` moves it into the base class: the public `fit` and `transform` are concrete methods that handle the flag and the error, and they call abstract `_fit` and `_transform` that subclasses implement. Subclasses cannot forget the check any more. Compare the two designs.
- **Abstract properties**: `@property` stacked on `@abstractmethod` forces subclasses to provide an attribute-like value.
- **`__slots__`** fixes the set of attributes an object may have (fewer typos, less memory). Rarely needed in DS code.
- **scikit-learn's version**: `sklearn.preprocessing.OneHotEncoder(handle_unknown="ignore")` together with `ColumnTransformer` and `Pipeline` is what you should use in production. Notebook X2 shows how to make your own transformers scikit-learn compatible.

---

## Sources and further reading

**Sources.** This notebook is based on:

- Python docs: [`abc`, abstract base classes](https://docs.python.org/3/library/abc.html) and [private variables in the tutorial](https://docs.python.org/3/tutorial/classes.html#private-variables) (why `_name` is a convention, not a lock).
- [scikit-learn: Common pitfalls, inconsistent preprocessing](https://scikit-learn.org/stable/common_pitfalls.html#inconsistent-preprocessing): the official description of the bug from notebook 00.
- scikit-learn: [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) and [Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html), what to use in production.

**For self-study.** Where to go next if you want to learn more:

- [Real Python: Python's property()](https://realpython.com/python-property/) (article, intermediate): validating setters and read-only attributes in depth.
- Luciano Ramalho, *Fluent Python*, 2nd edition, O'Reilly 2022 ([book site](https://www.fluentpython.com/)), chapter 13 "Interfaces, Protocols, and ABCs" (book, intermediate to advanced).
- In this repository: `src/churn/features.py` shows the template-method version of `Transformer`.

---

[← 03 · Special methods](03_special_methods.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [05 · Polymorphism, duck typing and protocols →](05_polymorphism_duck_typing_protocols.ipynb)